# Python Habits to Ditch & Idiomatic Replacements

### 🎯 Objective
Identify and replace obsolete or unpythonic code patterns with modern, idiomatic Python. 

By replacing manual resource management, string concatenation, and dangerous defaults with context managers, f-strings, `zip()`, and immutable default parameters, your code becomes safer, faster, and aligned with standard PEP 8 conventions.


### 📖 1. Common Antipatterns and Idiomatic Alternatives

#### 1. String Concatenation vs. F-Strings
Using the `+` operator for formatting creates redundant intermediate string objects and harms readability. Use formatted string literals (`f"..."`).


In [ ]:
# Antipattern
name = "Fulanito"
greeting_old = "Hello " + name + ", welcome!"

# Pythonic
greeting_new = f"Hello {name}, welcome!"

print(greeting_new)


#### 2. Manual Resource Cleanup vs. Context Managers
Leaving files to be manually closed risks file descriptor leaks if an unexpected exception occurs before `.close()`. Always use `with open(...)`.


In [ ]:
import tempfile

# Pythonic: Resource automatically closed even if an exception occurs
with tempfile.NamedTemporaryFile("w+", delete=True) as temp_file:
    temp_file.write("Hello from safe context manager!\n")
    temp_file.seek(0)
    print("Read content:", temp_file.read().strip())


#### 3. Index Looping vs. `zip()` and `enumerate()`
Avoid manually tracking indices when pairing two collections or needing an index alongside values.


In [ ]:
numbers = [1, 2, 3]
names = ["one", "two", "three"]

# Pythonic pairing with zip
for num, name in zip(numbers, names):
    print(f"Num: {num} -> Name: {name}")

# Pythonic index + value with enumerate
for idx, (num, name) in enumerate(zip(numbers, names), start=1):
    print(f"[{idx}] {name} is #{num}")


#### 4. Mutable Default Arguments
In Python, default arguments are evaluated once at function definition time, NOT at each call. A mutable default (`list`, `dict`, `set`) persists mutations across invocations.


In [ ]:
# Antipattern: def append_item(val, container=[]): container.append(val); return container

# Pythonic: Use None as sentinel and initialize inside function
def append_item(val: int, container: list[int] | None = None) -> list[int]:
    if container is None:
        container = []
    container.append(val)
    return container


print("First call:", append_item(1))
print("Second call:", append_item(2))  # Does NOT retain 1!


#### 5. Bare `except:` vs Specific Exceptions
Catching bare `except:` intercepts system exits (`KeyboardInterrupt`, `SystemExit`) and masks logic bugs. Always catch specific exceptions (`ValueError`, `KeyError`, etc.).


In [ ]:
def safe_int_conversion(val: str) -> int | None:
    try:
        return int(val)
    except (ValueError, TypeError):
        return None


print("Converted '42':", safe_int_conversion("42"))
print("Converted 'abc':", safe_int_conversion("abc"))


### 💻 2. Practice Challenge (Your Turn)

#### Challenge: Refactor Legacy Record Processor
You are handed a legacy data processor with multiple antipatterns:
1. It uses a mutable default argument `accumulator=[]`.
2. It concatenates strings with `+` instead of f-strings.
3. It accesses paired lists using manual `range(len(...))` indices instead of `zip()`.
4. It uses a bare `except:` that suppresses errors.
5. It checks `len(seq) == 0` instead of idiomatic truthiness checks (`not seq`).

Refactor `process_student_grades(names, scores, accumulator=None)` to adhere to modern Python standards.

**Requirements**:
- Accept `names: list[str]`, `scores: list[int | str]`, and `accumulator: list[str] | None = None`.
- If `accumulator` is `None`, initialize it as a fresh list.
- If `not names` or `not scores`, return the `accumulator` unchanged.
- Pair elements safely using `zip(names, scores)`.
- Convert score to integer. If score is invalid (cannot convert to `int`), catch `(ValueError, TypeError)` and record `f"{name.strip()}: INVALID"`.
- If valid, check if `score >= 70`:
  - Passed: `f"{name.strip()}: PASSED ({score}/100)"`
  - Failed: `f"{name.strip()}: FAILED ({score}/100)"`
- Append results to `accumulator` and return it.


In [ ]:
def process_student_grades(
    names: list[str],
    scores: list[int | str],
    accumulator: list[str] | None = None
) -> list[str]:
    """Refactored idiomatic grade processing pipeline."""
    # YOUR CODE HERE
    pass


# Automated Assertions
names1 = ["Alice", "Bob", "Charlie", "Diana"]
scores1 = [95, 62, "N/A", "88"]

res1 = process_student_grades(names1, scores1)
expected1 = [
    "Alice: PASSED (95/100)",
    "Bob: FAILED (62/100)",
    "Charlie: INVALID",
    "Diana: PASSED (88/100)"
]
assert res1 == expected1, f"Expected {expected1}, got {res1}"

# Test mutable default isolation
res2 = process_student_grades(["Eve"], [100])
assert res2 == ["Eve: PASSED (100/100)"]
assert len(res1) == 4, "State leaked between function calls!"

# Test custom accumulator injection
custom_acc = ["Header"]
res3 = process_student_grades(["Frank"], [75], accumulator=custom_acc)
assert res3 is custom_acc
assert res3 == ["Header", "Frank: PASSED (75/100)"]

# Test empty inputs
assert process_student_grades([], []) == []

print("✅ Challenge passed all test cases!")


<details>
<summary>💡 Hint</summary>

Initialize `if accumulator is None: accumulator = []`. Pair `zip(names, scores)`. Inside the loop, wrap the `int(score)` cast in a `try...except (ValueError, TypeError):` block, and format each string with f-strings.
</details>

<details>
<summary>🔍 Solution</summary>

```python
def process_student_grades(
    names: list[str],
    scores: list[int | str],
    accumulator: list[str] | None = None
) -> list[str]:
    if accumulator is None:
        accumulator = []

    if not names or not scores:
        return accumulator

    for name, raw_score in zip(names, scores):
        clean_name = name.strip()
        try:
            numeric_score = int(raw_score)
            status = "PASSED" if numeric_score >= 70 else "FAILED"
            accumulator.append(f"{clean_name}: {status} ({numeric_score}/100)")
        except (ValueError, TypeError):
            accumulator.append(f"{clean_name}: INVALID")

    return accumulator
```
</details>
